# Day 14 · 資料怎麼流：Data Handling 與 Dynamic Workflows

> 第三部・戰術編排　|　✅ 完整可執行

**前置需求**：🔑 需要 Gemini API 金鑰

**對應文章**：`Day 14 - 資料怎麼流：Data Handling 與 Dynamic Workflows.md`

## 今天要學會

1. 分辨 output / state 兩種資料傳遞方式
2. 用 Schema 約束節點之間的資料
3. `parameter_binding` 的兩種模式差在哪
4. 什麼時候該放棄畫圖、改用程式碼

## 環境設定

每一天都是獨立的，這段設定刻意重複，讓你可以從任何一天開始。

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

import google.adk

print("google-adk", google.adk.__version__)

google-adk 2.8.0


## 1. ⚠️ ADK 裡有**兩套**資料流機制

這是本日最重要、也最容易搞混的一件事。混用會拿到 `None` 或直接報錯。

| | Graph Workflow（Day 13） | Template Workflow Agent（Day 16） |
|---|---|---|
| 傳資料靠 | 節點參數綁定 | `output_key` 寫 state、`{key?}` 讀 |
| 結果放在 | **`event.output`** | `event.content` |
| 誰決定 | 邊 + `parameter_binding` | instruction 裡的 `{key?}` |

兩套都可以用 state，但**取值的方式完全不同**。

In [2]:
from google.adk import Workflow
from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.agents.context import Context
from google.adk.runners import InMemoryRunner
from google.adk.workflow import START, node
from google.genai import types


async def run_graph(wf, text: str):
    """跑一張圖，印出每個節點的 output。"""
    runner = InMemoryRunner(agent=wf, app_name="day14")
    sid = await new_session(runner)
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    outs = []
    async for ev in runner.run_async(user_id="student", session_id=sid, new_message=msg):
        o = getattr(ev, "output", None)
        if o is not None:
            print(f"  ▪ [output] {o}")
            outs.append(o)
        elif ev.is_final_response() and ev.content:
            t = "".join(p.text or "" for p in ev.content.parts if p.text)
            if t:
                print(f"  💬 [content] {t.strip()[:100]}")
    return outs

## 2. `parameter_binding`：節點參數從哪裡來

`@node` 裝飾器有一個 `parameter_binding` 參數，兩種模式：

| 模式 | 從哪裡找參數 | 預設 |
|---|---|---|
| `"state"` | session state | ✅ |
| `"node_input"` | **上一個節點的回傳值** | |

**兩種都是用「參數名稱」去對應**，差別只在去哪裡找。

In [3]:
import inspect

print("node 裝飾器:", inspect.signature(node).parameters["parameter_binding"])

node 裝飾器: parameter_binding: "Literal['state', 'node_input']" = 'state'


### 模式一：`state`（預設）

In [4]:
@node
def make_order() -> dict:
    return {"order_id": "A-1", "amount": 1200}


extractor = LlmAgent(
    name="extractor",
    model=get_model(),
    instruction="從使用者訊息抽出商品名稱，只回商品名稱四個字以內。",
    output_key="product",      # ← 寫進 state
)


@node
def from_state(product: str) -> str:
    """參數名 product 對應 state["product"]。"""
    return f"[state 綁定] 商品 = {product}"


state_flow = Workflow(
    name="state_flow",
    edges=[(START, extractor), (extractor, from_state)],
)
await run_graph(state_flow, "我要買一支藍芽耳機")

  💬 [content] 藍芽耳機
  ▪ [output] [state 綁定] 商品 = 藍芽耳機


['[state 綁定] 商品 = 藍芽耳機']

### 模式二：`node_input`

In [5]:
@node
def produce() -> dict:
    return {"order_id": "B-2", "amount": 8800}


@node(parameter_binding="node_input")
def from_node_input(order_id: str, amount: int) -> str:
    """參數名對應「上一個節點回傳的 dict」裡的 key。"""
    return f"[node_input 綁定] {order_id} 金額 {amount}"


input_flow = Workflow(
    name="input_flow",
    edges=[(START, produce), (produce, from_node_input)],
)
await run_graph(input_flow, "go")

  ▪ [output] {'order_id': 'B-2', 'amount': 8800}
  ▪ [output] [node_input 綁定] B-2 金額 8800


[{'order_id': 'B-2', 'amount': 8800}, '[node_input 綁定] B-2 金額 8800']

### 📌 兩種模式都會這樣失敗

參數名對不上時，錯誤訊息會告訴你它去哪裡找過：

In [6]:
@node
def wrong_state(no_such_key: str) -> str:
    return no_such_key


@node(parameter_binding="node_input")
def wrong_input(no_such_key: str) -> str:
    return no_such_key


for label, slug, bad_node in (
    ("state 模式", "state", wrong_state),
    ("node_input 模式", "node_input", wrong_input),
):
    # 注意：Workflow / node 的 name 必須是合法的 Python 識別字，
    # 不能有空白或中文——所以這裡用 slug 而不是 label。
    wf = Workflow(name=f"bad_{slug}", edges=[(START, produce), (produce, bad_node)])
    try:
        await run_graph(wf, "go")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}")
        print(f"  {str(exc)[:150]}")

  ▪ [output] {'order_id': 'B-2', 'amount': 8800}
state 模式: ValueError
  Missing value for parameter "no_such_key" of function "wrong_state". It was not found in state and has no default value.
  ▪ [output] {'order_id': 'B-2', 'amount': 8800}
node_input 模式: ValueError
  Missing value for parameter "no_such_key" of function "wrong_input". It was not found in node_input and has no default value.


注意錯誤訊息的差別：一個說 `not found in state`，
一個說 `not found in node_input`。**這是你判斷自己用錯模式的最快方法。**

## 3. 用 Schema 約束節點之間的資料

dict 傳來傳去很容易打錯 key。用 Pydantic 把介面定死：

In [7]:
from pydantic import BaseModel, Field


class FlightQuery(BaseModel):
    origin: str = Field(description="出發城市")
    destination: str = Field(description="目的地城市")
    passengers: int = Field(description="人數")


class FlightResult(BaseModel):
    airline: str
    price_twd: int
    seats_left: int


parse_query = LlmAgent(
    name="parse_query",
    model=get_model(),
    instruction="從使用者的訂票需求中抽出出發地、目的地與人數。",
    output_schema=FlightQuery,
    output_key="query",
)


@node
def search_flight(query: dict) -> dict:
    """用 schema 驗證輸入，再回傳同樣經過驗證的輸出。"""
    q = FlightQuery.model_validate(query)          # ← 進來時驗證
    result = FlightResult(
        airline=f"{q.origin[:1]}{q.destination[:1]}-{100 + q.passengers}",
        price_twd=14800 * q.passengers,
        seats_left=9,
    )
    return result.model_dump()                     # ← 出去時也是定義好的形狀


@node
def render(query: dict) -> str:
    q = FlightQuery.model_validate(query)
    return f"✅ {q.origin} → {q.destination}，{q.passengers} 位"


schema_flow = Workflow(
    name="schema_flow",
    edges=[(START, parse_query), (parse_query, search_flight), (search_flight, render)],
)
await run_graph(schema_flow, "我要從台北飛東京，兩個人")

  💬 [content] {
  "origin": "台北",
  "destination": "東京",
  "passengers": 2
}
  ▪ [output] {'airline': '台東-102', 'price_twd': 29600, 'seats_left': 9}
  ▪ [output] ✅ 台北 → 東京，2 位


[{'airline': '台東-102', 'price_twd': 29600, 'seats_left': 9}, '✅ 台北 → 東京，2 位']

### 為什麼值得多寫這幾行

沒有 schema 時，上游少給一個 key，下游要到**執行到那一行**才 `KeyError`。
有 schema 時，`model_validate()` 會**在節點入口就擋下來**，
而且錯誤訊息告訴你缺什麼欄位。

In [8]:
try:
    FlightQuery.model_validate({"origin": "台北", "destination": "東京"})  # 少了 passengers
except Exception as exc:
    print("Schema 擋下來的錯誤：")
    print(str(exc)[:300])

Schema 擋下來的錯誤：
1 validation error for FlightQuery
passengers
  Field required [type=missing, input_value={'origin': '台北', 'destination': '東京'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


## 4. State 的作用域前綴在圖裡一樣有效

Day 09 講的 `user:` / `app:` / `temp:` 前綴，在圖的節點裡照樣可用。

In [9]:
@node
def write_scopes(ctx: Context) -> str:
    ctx.state["plain"] = "只有這個 session"
    ctx.state["user:tier"] = "gold"          # 跨 session
    ctx.state["app:region"] = "TW"           # 跨使用者
    ctx.state["temp:secret"] = "不會被存"     # 不會寫進 session
    return "已寫入四種作用域"


@node
def read_scopes(ctx: Context) -> str:
    # ⚠️ ctx.state 是 State 物件，不是 dict——沒有 .keys() / .items()。
    #    要拿全部內容得用 to_dict()。
    keys = sorted(ctx.state.to_dict())
    return f"節點看得到的 key: {keys}"


scope_flow = Workflow(
    name="scope_flow",
    edges=[(START, write_scopes), (write_scopes, read_scopes)],
)
await run_graph(scope_flow, "go")

  ▪ [output] 已寫入四種作用域
  ▪ [output] 節點看得到的 key: ['app:region', 'plain', 'temp:secret', 'user:tier']


['已寫入四種作用域', "節點看得到的 key: ['app:region', 'plain', 'temp:secret', 'user:tier']"]

### ⚠️ `ctx.state` 不是 dict

它是一個 `State` 物件。支援 `[]`、`in`、`get()`，
**但沒有 `.keys()` / `.items()` / `.values()`**——寫了會 `AttributeError`。

In [10]:
from google.adk.sessions.state import State

print("State 的公開 API:")
print(" ", [m for m in dir(State) if not m.startswith("_")])

demo = State(value={"a": 1, "user:b": 2}, delta={"c": 3})
print()
print("  demo['a']        =", demo["a"])
print("  'a' in demo      =", "a" in demo)
print("  demo.get('zzz')  =", demo.get("zzz"))
print("  demo.to_dict()   =", demo.to_dict())
try:
    demo.keys()
except AttributeError as exc:
    print(f"  demo.keys()      → ❌ {exc}")

State 的公開 API:
  ['APP_PREFIX', 'TEMP_PREFIX', 'USER_PREFIX', 'get', 'has_delta', 'setdefault', 'to_dict', 'update']

  demo['a']        = 1
  'a' in demo      = True
  demo.get('zzz')  = None
  demo.to_dict()   = {'a': 1, 'user:b': 2, 'c': 3}
  demo.keys()      → ❌ 'State' object has no attribute 'keys'


注意 `to_dict()` 會把 **value 和尚未 commit 的 delta 合起來**回傳——
這也是為什麼它才是拿「目前完整狀態」的正確方法。

## 5. ⚠️ state 不要放大東西

整張圖共用同一份 state，而且它會跟著每次模型呼叫走。
塞大檔案進去 = 每個節點都在付那份資料的錢。

**正確做法**：大東西存 Artifact（Day 11），state 只放**指標**。

In [11]:
@node
def bad_practice(ctx: Context) -> str:
    ctx.state["report_content"] = "x" * 50_000     # ❌ 50KB 塞進 state
    return "已存入 state（不好的做法）"


@node
def good_practice(ctx: Context) -> str:
    # ✅ 大東西存 artifact，state 只留檔名
    ctx.state["report_ref"] = "user:report.md"
    return "state 只存指標（好的做法）"


for label, n in (("❌ 直接塞 state", bad_practice), ("✅ 只存指標", good_practice)):
    wf = Workflow(name=f"sz_{n.name}", edges=[(START, n)])
    runner = InMemoryRunner(agent=wf, app_name="day14")
    sid = await new_session(runner)
    msg = types.Content(role="user", parts=[types.Part(text="go")])
    async for _ in runner.run_async(user_id="student", session_id=sid, new_message=msg):
        pass
    st = await peek_state(runner, sid)
    size = sum(len(str(v)) for v in st.values())
    print(f"{label}: state 總大小 {size:,} 字元")

❌ 直接塞 state: state 總大小 50,000 字元
✅ 只存指標: state 總大小 14 字元


## 6. 什麼時候該放棄畫圖

圖很好用，但不是萬能。這是原文提到、值得展開的判斷準則：

| 情況 | 建議 |
|---|---|
| 節點數固定、流程可畫在白板上 | ✅ 用圖 |
| 有明確的條件分支 | ✅ 用圖（Day 13） |
| **分支數量在執行時才知道** | ❌ 改用程式碼迴圈 |
| **要處理不定長度的清單** | ❌ 改用程式碼 |
| 需要遞迴 / 動態產生節點 | ❌ 改用程式碼 |

判準很簡單：

> **圖的結構必須在「執行前」就決定好。**
> 結構取決於資料時，就不該用圖。

### 動態流程用一般的 async 程式碼就好

例如「有幾筆訂單就跑幾次審核」——訂單筆數執行時才知道，畫不出圖：

In [12]:
import asyncio

reviewer = LlmAgent(
    name="reviewer",
    model=get_model(),
    instruction="判斷這筆支出是否合理，只回『合理』或『需複核』加一句理由，繁體中文。",
)

ORDERS = [
    "計程車費 320 元，客戶拜訪",
    "團隊聚餐 18000 元，5 人",
    "文具採購 450 元",
]


async def review_all(items: list[str]) -> list[str]:
    """筆數不固定，用一般程式碼並行處理——不需要圖。"""
    return await asyncio.gather(*[run_once(reviewer, it) for it in items])


results = await review_all(ORDERS)
for item, verdict in zip(ORDERS, results):
    print(f"  • {item}")
    print(f"    → {verdict.strip()[:70]}")

  • 計程車費 320 元，客戶拜訪
    → 合理，客戶拜訪屬正常公務交通支出。
  • 團隊聚餐 18000 元，5 人
    → 合理，每人平均 3600 元，符合團隊聚餐的正常開銷範圍。
  • 文具採購 450 元
    → 合理，金額小且符合日常辦公需求。


### 混合用法才是常態

實務上通常是：

```
  Workflow（固定的主流程、有分支）
     └─ 某個節點內部用一般程式碼處理不定長度的清單
```

節點內部可以做任何事，包括跑迴圈、開並行、呼叫其他 agent。

In [13]:
@node
async def review_batch(ctx: Context) -> str:
    """節點內部用程式碼處理不定長度的資料。"""
    verdicts = await review_all(ORDERS)
    flagged = [o for o, v in zip(ORDERS, verdicts) if "需複核" in v]
    ctx.state["flagged_count"] = len(flagged)
    return f"審了 {len(ORDERS)} 筆，其中 {len(flagged)} 筆需複核"


@node
def notify(flagged_count: int) -> str:
    return ("✅ 全部通過，直接入帳" if flagged_count == 0
            else f"📋 有 {flagged_count} 筆需要人工複核")


hybrid_flow = Workflow(
    name="hybrid_flow",
    edges=[(START, review_batch), (review_batch, notify)],
)
await run_graph(hybrid_flow, "開始審核")

  ▪ [output] 審了 3 筆，其中 0 筆需複核
  ▪ [output] ✅ 全部通過，直接入帳


['審了 3 筆，其中 0 筆需複核', '✅ 全部通過，直接入帳']

## 7. 常見錯誤與踩坑

| 症狀 | 原因 |
|---|---|
| `not found in state` | 用預設的 state 綁定，但上游沒設 `output_key` |
| `not found in node_input` | 用了 `parameter_binding="node_input"`，但參數名對不上上游回傳的 key |
| 節點拿到 `None` | 混用了兩套資料流機制（`event.output` vs `output_key`） |
| 下游 `KeyError` | 沒用 schema 驗證，上游少給欄位 |
| 圖越跑越慢 | state 塞了大東西，改存 Artifact |
| `'State' object has no attribute 'keys'` | `ctx.state` 不是 dict，用 `to_dict()` |
| `Node name '...' must be a valid Python identifier` | Workflow / node 的 name 不能有空白或中文 |
| 想依資料筆數動態產生節點 | **圖做不到**，改用一般程式碼 |

## 8. 動手練習

1. 把 `from_state` 的參數改名成 `item`，重跑，確認錯誤訊息說 `not found in state`。
2. 把 `from_node_input` 的 `parameter_binding` 拿掉（改回預設），
   看錯誤訊息從 `node_input` 變成 `state`。
3. 幫 `search_flight` 的輸出加一個 `FlightResult` 沒有的欄位，
   確認 `model_validate` 會不會擋（提示：Pydantic 預設允許多餘欄位嗎？）。
4. 把 `review_batch` 改成序列處理，比較耗時差異。

## 本日回顧

- **⚠️ ADK 有兩套資料流機制**：圖用節點參數綁定 + `event.output`；
  workflow agent 用 `output_key` + `{key?}` + `event.content`。混用會出事。
- **`parameter_binding` 兩種模式**：`state`（預設，從 session state 找）
  與 `node_input`（從上一個節點的回傳值找）。**兩者都用參數名稱對應**，
  錯誤訊息會明講它去哪裡找過。
- **用 Pydantic schema 把節點介面定死**，錯誤在入口就擋下來。
- **`ctx.state` 是 `State` 物件不是 dict**：有 `[]`、`in`、`get()`、`to_dict()`，
  **沒有 `.keys()`**。
- **state 只放指標，大東西存 Artifact**（Day 11）。
- **圖的結構必須在執行前決定**；結構取決於資料時，改用一般程式碼——
  實務上最常見的是「圖負責主流程，節點內部用程式碼處理不定長度的資料」。

---
**下一天 → `../day15_human_in_the_loop/`**